In [2]:
import json
from tqdm import tqdm
file_path = "instruction-data-with-response.json"
with open(file_path, "r") as file:
    test_data = json.load(file)
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = (
        f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    )
    return instruction_text + input_text

In [5]:
import urllib.request

def query_model(prompt, model="llama3", url="http://localhost:11434/api/chat"):
    data = { #Create the data payload as a dictionary
        "model": model,
        "messages":[
            {"role" : "user", "content" : prompt}
        ],
        "options":{ #Setting deteministic responses
            "seed": 123,
            "temperature": 0,
            "num_context": 2048
        }
    }

    payload = json.dumps(data).encode("utf-8") #Converts the dictionary to a json formatted string and encodes it to bytes
    request = urllib.request.Request(
        url,
        data=payload,
        method="POST"
    )

    request.add_header("Content-Type", "application/json")  #Creates a request object, setting the method to POST and adding necessary headers

    response_data = ""

    with urllib.request.urlopen(request) as response:
        while True:
            line = response.readline().decode("utf-8")
            if not line:
                break
            response_json = json.loads(line)
            response_data += response_json["message"]["content"]

    return response_data



In [6]:
#Sanity Check
#Run the ollama in the background

model="llama3"
result = query_model("What do Llamas eat?", model=model)
print(result)

Llamas are herbivores, which means they primarily eat plants and plant-based foods. Their diet typically consists of:

1. Grasses: Llamas love to graze on various types of grasses, including tall grasses, short grasses, and even weeds.
2. Hay: Hay, such as alfalfa or timothy hay, is a staple in a llama's diet. It provides essential fiber and nutrients.
3. Grains: Llamas may be fed grains like oats, barley, or corn as a supplement to their diet.
4. Fruits and vegetables: Llamas enjoy fruits and vegetables like apples, carrots, and sweet potatoes as treats or as part of their regular diet.
5. Leaves: Llamas will also eat leaves from trees and shrubs, such as willow, alder, or cedar.
6. Bark: In the winter or when other food sources are scarce, llamas may eat the bark of trees, like aspen or birch.

In the wild, llamas will roam freely and eat whatever is available in their natural habitat. In captivity, their diet is typically formulated to meet their nutritional needs, and they may rece

In [11]:
for entry in test_data[:3]:
    prompt = (
        f"Given the input '{format_input(entry)}'"
        f"and correct output '{entry['output']}',"
        f"score the model response '{entry['model_response']}'"
        f" on a scale from 0 to 100, where 100 is the best score. Only output Score without any additional text or explanation."
    )

    print("\nDataset response:")
    print(">>", entry['output'])
    print("\nDataset response:")
    print(">>", entry['model_response'])
    print("\nScore:")
    print(">>", query_model(prompt))
    print("\n------------------------------\n")



Dataset response:
>> The car is as fast as lightning.

Dataset response:
>> The car is as fast as a horse.

Score:
>> 85

------------------------------


Dataset response:
>> The type of cloud typically associated with thunderstorms is cumulonimbus.

Dataset response:
>> A type of cloud typically associated with thunderstorms is the active component of the active component of the active component of the active component of the active component of the active component of the active component of the active component of the active component of the active component of the active component of the active component of the active component of the active component of the active component of the active component of the active component of the active component of the active component of the active component of the active component of the active component of the active component of the active component of the active component of the active component of the active component of the active componen

In [16]:
def generate_model_scores(json_data, json_key, model="llama3"):
    scores = []
    for entry in tqdm(json_data, desc="Scoring entries"):
        prompt = (
                    f"Given the input `{format_input(entry)}` "
                    f"and correct output `{entry['output']}`, "
                    f"score the model response `{entry[json_key]}`"
                    f" on a scale from 0 to 100, where 100 is the best score. "
                    f"Respond with the integer number only." 
        )

        score = query_model(prompt, model)

        try:
            scores.append(int(score))
        except ValueError:
            print(f"Warning: Could not convert score '{score}' to int for entry: {entry}")

    return scores


In [17]:
scores = generate_model_scores(test_data, "model_response", model="llama3")

print(f"Number of scores: {len(scores)} out of {len(test_data)}")
print(f"Average score : {sum(scores)/len(scores):.2f}\n")

Scoring entries:  72%|███████▏  | 79/110 [00:52<00:26,  1.18it/s]


### Input: 
Swim: Verb
Beautiful: Adjective
Quickly: Adverb

### Score: 100' to int for entry: {'instruction': 'Classify the following words by their grammatical categories: swim, beautiful, quickly', 'input': '', 'output': 'Swim: Verb\nBeautiful: Adjective\nQuickly: Adverb', 'model_response': '### Input: \nSwim, beautiful\n\n\nThe following words are classified as being either formal or informal.'}


Scoring entries: 100%|██████████| 110/110 [01:15<00:00,  1.46it/s]

Number of scores: 109 out of 110
Average score : 32.17

